# 06. 연구 확장: 환자 교차검증과 공식 예제

기존 두 test 환자(HCC02/HCC09)는 제외하고 8명의 개발 환자만 사용합니다. 4개 outer fold에서 각 환자는 한 번만 평가됩니다. 내부 validation으로 head 설정을 고른 뒤, 해당 outer fold의 나머지 환자에 다시 학습합니다. 개발 결과를 반복 확인하므로 새로운 blind test가 아닙니다.

기본 동작은 이미 실행된 작은 결과를 읽습니다. 대규모 추출이나 전체 학습을 자동 실행하지 않습니다. 세포 분류, 공식 MS 예제, 실제 섭동 반응 예측은 서로 다른 task입니다.


In [1]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/research/protocol_annotation.yaml').exists()
REPORT = ROOT / 'reports/research_phase2'
pd.read_csv(REPORT / 'annotation_pilot_summary/pooled_metrics.tsv', sep='\t')


,model,n_cells,pooled_macro_f1,balanced_accuracy,mean_fold_macro_f1,development_only
0,pca,8760,0.938026,0.952574,0.932742,True
1,scgpt,8760,0.940900,0.951443,0.934294,True
2,geneformer,8760,0.947063,0.946498,0.941677,True


## 어떤 세포에서 오차가 나는가?

pooled macro-F1은 fold 점수의 단순 평균과 다릅니다. 아래에는 8명 전체의 class별 support와 donor별 결과를 함께 봅니다. 특정 환자에게 없는 class도 공통 6개 class 점수에서는 0으로 처리되므로 support와 함께 해석합니다.


In [2]:
pd.read_csv(REPORT / 'annotation_pilot_summary/per_class.tsv', sep='\t')


,model,label,precision,recall,f1-score,support
0,pca,B,0.780362,0.917933,0.843575,329.0
1,pca,Endothelial,0.929435,0.954451,0.941777,483.0
2,pca,Fibroblast,0.942373,0.958621,0.950427,290.0
3,pca,Hepatocyte,0.942586,0.968180,0.955211,2357.0
4,pca,Myeloid,0.970487,0.969533,0.970010,2035.0
5,pca,T/NK,0.988491,0.946724,0.967157,3266.0
6,scgpt,B,0.835655,0.911854,0.872093,329.0
7,scgpt,Endothelial,0.922000,0.954451,0.937945,483.0
8,scgpt,Fibroblast,0.925926,0.948276,0.936968,290.0
9,scgpt,Hepatocyte,0.946866,0.967756,0.957197,2357.0


In [3]:
pd.read_csv(REPORT / 'annotation_pilot_summary/by_donor.tsv', sep='\t')


,model,patient_id,n_cells,macro_f1_common_ontology,n_errors
0,pca,HCC01,472,0.920872,20
1,pca,HCC03,1032,0.896811,56
2,pca,HCC04,958,0.952256,46
3,pca,HCC05,1113,0.953956,35
4,pca,HCC06,1219,0.906370,62
5,pca,HCC07,846,0.906563,47
6,pca,HCC08,1909,0.935904,67
7,pca,HCC10,1211,0.944844,39
8,scgpt,HCC01,472,0.872666,23
9,scgpt,HCC03,1032,0.900721,57


## 공식 scGPT MS 예제

공식 reference/query 파일, seed 0, 10 epoch, batch size 32를 유지했습니다. 현재 장비에서는 non-flash attention을 사용하며, 공식 weight-key 변환 후 attention 24개 텐서가 일치하는지 확인했습니다. 이 설정 차이 때문에 동일한 수치의 완전 재현으로 단정하지 않습니다. 64-cell smoke 점수와 전체 실행 점수를 구별합니다.


In [4]:
official = json.loads((REPORT / 'runs/R00_ms_full_seed0_v2/run.json').read_text())
print('status:', official['reproduction_status'])
print('reference/query:', official['n_reference'], official['n_evaluation'])
print('metrics:', official['metrics'])
print('training seconds:', official['wall_training_seconds'])
print('GPU GiB:', official['peak_process_gpu_gib'])


status: modified_reproduction
reference/query: 7844 13468
metrics: {'test/accuracy': 0.8635283635283635, 'test/precision': 0.7153100418139433, 'test/recall': 0.6944365520156915, 'test/macro_f1': 0.6755809331791585}
training seconds: 1476.0225419513881
GPU GiB: 9.80078125


## 원할 때 개발 fold 하나 다시 실행

아래 `RUN_ONE_FOLD`를 True로 바꾸면 이미 저장된 scGPT 파일럿 표현에 대해 CPU로 head를 다시 학습합니다. 새로운 run ID를 사용하고 기존 결과를 덮어쓰지 않습니다. 새 count/embedding 추출, 외부 cohort 평가, 4개 fold 전체 실행은 하지 않습니다.


In [5]:
RUN_ONE_FOLD = False
if RUN_ONE_FOLD:
    from datetime import datetime, timezone
    run_id = 'tutorial06_scgpt_fold0_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    subprocess.run([sys.executable, str(ROOT / 'scripts/research/run_annotation_cv.py'),
                    '--config', str(ROOT / 'configs/research/protocol_annotation.yaml'),
                    '--model', 'scgpt', '--fold', '0', '--run-id', run_id], cwd=ROOT, check=True)
    display(pd.read_json(REPORT / 'runs' / run_id / 'metrics.json', typ='series'))
else:
    print('저장 결과 읽기 모드: 새 학습을 실행하지 않았습니다.')


저장 결과 읽기 모드: 새 학습을 실행하지 않았습니다.


## 다음 연구 결과의 범위

63,036개 전체 개발 세포의 frozen 특징 추출은 완료됐습니다. 전체 donor CV, scVI, LiverCT query adaptation, 실제 GEARS 반응 예측, 데이터 개선과 LoRA 대조 실험은 각각 실행 결과가 생긴 뒤 이 튜토리얼에 연결합니다. 외부 HCC 최종 cohort는 모델·데이터 선택을 고정한 뒤에만 평가합니다.
